# JHTDB Data Fetching Pipeline - High-Resolution (512×512)

This notebook fetches high-resolution channel-flow snapshots from the Johns Hopkins Turbulence Database (JHTDB) using the `givernylocal` library with an upgraded API token.

**NEW: High-capacity token allows up to 500,000 points per query!**
- Previous limit: 4,096 points (required tiling for 256×256)
- New limit: 500,000 points (allows direct 512×512 queries!)
- 512×512 = 262,144 points ✓ (well under limit)

**Key features:**
- **Single-query fetching** - No tiling needed for 512×512!
- Detailed progress tracking for each snapshot
- Fetches velocity components (ux, uy, uz) and computes vorticity
- Saves each snapshot as individual `.npz` files in `data/raw/channel_sr_512/`
- Creates visualizations for each snapshot
- Converts all `.npz` files to HDF5 format
- Generates animations for all components

**Output structure:**
```
data/raw/channel_sr_512/
├── channel_x-z_y0.9_t{time}.npz    # Individual snapshots (512×512)
├── visualizations/
│   ├── snapshot_t{time}.png        # Snapshot visualizations
│   ├── animation_ux.gif            # Animations
│   ├── animation_uy.gif
│   ├── animation_uz.gif
│   └── animation_omega_z.gif
└── channel_sr_512.h5                # Combined HDF5 file
```

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
def find_project_root(start: Path, target="models", max_up: int = 8) -> Path:
    root = start.resolve()
    for _ in range(max_up):
        if (root / target).exists():
            return root
        if root.parent == root:
            break
        root = root.parent
    raise FileNotFoundError(f"Could not find '{target}' above {start}")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root added to sys.path:", ROOT)

from typing import Tuple, List
import warnings
from datetime import datetime
import glob
import yaml

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from tqdm.notebook import tqdm
from IPython.display import display, clear_output, HTML

from givernylocal.turbulence_dataset import turb_dataset
from givernylocal.turbulence_toolkit import getData

from utils.io import load_env_token
from utils.physics import compute_vorticity_2d

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.facecolor'] = 'white'

print("✓ Libraries imported successfully")

## Configuration

**Load configuration from YAML file** to easily switch between different datasets and planes.

Available configurations in `configs_jhtdb/`:
- `channel_xz_wall_parallel.yaml` - Wall-parallel (x-z) plane at y=0.9 **(default)**
- `channel_xy_wall_normal.yaml` - Wall-normal (x-y) plane at z=1.5π
- `channel_yz_wall_normal.yaml` - Wall-normal (y-z) plane at x=π
- `isotropic_xy_plane.yaml` - Forced isotropic turbulence (x-y plane)

In [ ]:
# =============================================================================
# SELECT CONFIGURATION FILE
# =============================================================================
# Change this line to switch between different datasets/planes:

#CONFIG_FILE = "channel_xy_wall_normal.yaml"  
CONFIG_FILE = "isotropic_xy_plane.yaml"

# Other options:
# CONFIG_FILE = "channel_xy_wall_normal.yaml"
# CONFIG_FILE = "channel_yz_wall_normal.yaml"
# CONFIG_FILE = "isotropic_xy_plane.yaml"

# =============================================================================

# Load configuration
ROOT_DIR = ROOT
config_path = ROOT_DIR / "configs_jhtdb" / CONFIG_FILE

print(f"Loading configuration: {CONFIG_FILE}")
print(f"  Path: {config_path}")

if not config_path.exists():
    raise FileNotFoundError(f"Configuration file not found: {config_path}")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"✓ Configuration loaded successfully")
print(f"  Experiment: {config['experiment']['name']}")
print(f"  Description: {config['experiment']['description']}\n")

# =============================================================================
# EXTRACT PARAMETERS FROM CONFIG
# =============================================================================

# Dataset configuration
DATASET = config['dataset']['name']
VARIABLE = config['dataset']['variable']

# Plane configuration
plane_config = config['plane']
plane_type = plane_config['type']
plane_orientation = plane_config['orientation']
fixed_coord_name = plane_config['fixed_coordinate']['name']
fixed_coord_value = plane_config['fixed_coordinate']['value']

# Grid configuration
GRID_SIZE = config['domain']['grid_size']

# Determine which coordinates to use based on plane orientation
if plane_orientation == 'xz':
    # Wall-parallel x-z plane (fixed y)
    X_MIN = config['domain']['x']['min']
    X_MAX = config['domain']['x']['max']
    Z_MIN = config['domain']['z']['min']
    Z_MAX = config['domain']['z']['max']
    Y_LOCATION = fixed_coord_value
    coord1_name, coord2_name = 'x', 'z'
    
elif plane_orientation == 'xy':
    # Wall-normal x-y plane (fixed z)
    X_MIN = config['domain']['x']['min']
    X_MAX = config['domain']['x']['max']
    Y_MIN = config['domain']['y']['min']
    Y_MAX = config['domain']['y']['max']
    Z_LOCATION = fixed_coord_value
    coord1_name, coord2_name = 'x', 'y'
    
elif plane_orientation == 'yz':
    # Wall-normal y-z plane (fixed x)
    Y_MIN = config['domain']['y']['min']
    Y_MAX = config['domain']['y']['max']
    Z_MIN = config['domain']['z']['min']
    Z_MAX = config['domain']['z']['max']
    X_LOCATION = fixed_coord_value
    coord1_name, coord2_name = 'y', 'z'

# Temporal configuration
NUM_SNAPSHOTS = config['temporal']['num_snapshots']
TIME_START = config['temporal']['time_start']
TIME_END = config['temporal']['time_end']
TIMES = np.linspace(TIME_START, TIME_END, NUM_SNAPSHOTS)

# Interpolation settings
TEMPORAL_METHOD = config['interpolation']['temporal_method']
SPATIAL_METHOD = config['interpolation']['spatial_method']
SPATIAL_OPERATOR = config['interpolation']['spatial_operator']

# Output configuration
OUTPUT_DIR = ROOT_DIR / config['output']['base_dir'] / config['output']['experiment_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR = OUTPUT_DIR / config['output']['visualization_dir']
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Visualization settings
VIZ_COLORMAPS = config['visualization']['colormaps']
SNAPSHOT_FREQUENCY = config['visualization']['snapshot_frequency']

# =============================================================================
# QUERY CAPACITY CHECK
# =============================================================================

TOTAL_POINTS = GRID_SIZE * GRID_SIZE
MAX_ALLOWED = 500000

print(f"Query capacity check:")
print(f"  Requested: {GRID_SIZE}×{GRID_SIZE} = {TOTAL_POINTS:,} points")
print(f"  Max allowed: {MAX_ALLOWED:,} points")
print(f"  Status: {'✓ OK' if TOTAL_POINTS <= MAX_ALLOWED else '✗ EXCEEDS LIMIT'}")
print(f"  Utilization: {100 * TOTAL_POINTS / MAX_ALLOWED:.1f}%\n")

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print(f"Configuration Summary:")
print(f"  Dataset: {DATASET}")
print(f"  Plane type: {plane_type}")
print(f"  Plane orientation: {plane_orientation} ({coord1_name}-{coord2_name})")
print(f"  Fixed coordinate: {fixed_coord_name} = {fixed_coord_value:.4f}")
print(f"  Grid: {GRID_SIZE}×{GRID_SIZE}")

if plane_orientation == 'xz':
    print(f"  Domain: x∈[{X_MIN:.4f}, {X_MAX:.4f}], z∈[{Z_MIN:.4f}, {Z_MAX:.4f}]")
elif plane_orientation == 'xy':
    print(f"  Domain: x∈[{X_MIN:.4f}, {X_MAX:.4f}], y∈[{Y_MIN:.4f}, {Y_MAX:.4f}]")
elif plane_orientation == 'yz':
    print(f"  Domain: y∈[{Y_MIN:.4f}, {Y_MAX:.4f}], z∈[{Z_MIN:.4f}, {Z_MAX:.4f}]")

print(f"  Snapshots: {NUM_SNAPSHOTS} from t={TIME_START} to t={TIME_END}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Visualization directory: {VIZ_DIR}")

## Authentication

Connect to JHTDB using your API token. The token should be stored in a `.env` file:
```
JHTDB_API_TOKEN=your-token-here
```

In [ ]:
# Load NEW high-capacity API token
token = load_env_token('JHTDB_MY_TOKEN', ROOT / '.env')
if token is None:
    # Fallback to old token
    token = load_env_token('JHTDB_API_TOKEN', ROOT / '.env')
    if token is None:
        raise RuntimeError(
            "JHTDB token not found. Please create a .env file with:\n"
            "JHTDB_MY_TOKEN=your-new-high-capacity-token-here"
        )
    print("⚠ Using old token (4096 point limit)")
else:
    print("✓ Using NEW high-capacity token (500,000 point limit)")

# Initialize dataset
dataset = turb_dataset(
    dataset_title=DATASET,
    output_path=str(ROOT_DIR / "data/giverny_cache"),
    auth_token=token
)

print(f"✓ Connected to JHTDB dataset '{DATASET}'")
print(f"  Token: {token[:20]}...")

## Grid Setup

Create the spatial grid for high-resolution queries. With the new token, **no tiling is needed** - we can fetch the entire 512×512 plane in a single query!

In [ ]:
# Create 1D coordinate arrays based on plane orientation
if plane_orientation == 'xz':
    coord1_coords = np.linspace(X_MIN, X_MAX, GRID_SIZE, dtype=np.float64)
    coord2_coords = np.linspace(Z_MIN, Z_MAX, GRID_SIZE, dtype=np.float64)
    fixed_coord = Y_LOCATION
    coord1_label, coord2_label = 'x', 'z'
    
elif plane_orientation == 'xy':
    coord1_coords = np.linspace(X_MIN, X_MAX, GRID_SIZE, dtype=np.float64)
    coord2_coords = np.linspace(Y_MIN, Y_MAX, GRID_SIZE, dtype=np.float64)
    fixed_coord = Z_LOCATION
    coord1_label, coord2_label = 'x', 'y'
    
elif plane_orientation == 'yz':
    coord1_coords = np.linspace(Y_MIN, Y_MAX, GRID_SIZE, dtype=np.float64)
    coord2_coords = np.linspace(Z_MIN, Z_MAX, GRID_SIZE, dtype=np.float64)
    fixed_coord = X_LOCATION
    coord1_label, coord2_label = 'y', 'z'

# Calculate grid spacing for vorticity computation
d_coord1 = coord1_coords[1] - coord1_coords[0]
d_coord2 = coord2_coords[1] - coord2_coords[0]

print(f"Grid setup:")
print(f"  {coord1_label}: {GRID_SIZE} points, spacing d{coord1_label}={d_coord1:.6f}")
print(f"  {coord2_label}: {GRID_SIZE} points, spacing d{coord2_label}={d_coord2:.6f}")
print(f"  Fixed {fixed_coord_name}: {fixed_coord:.4f}")
print(f"  Total points: {GRID_SIZE * GRID_SIZE:,}")
print(f"  ✓ Single query per snapshot (no tiling needed!)")

## Data Fetching Functions

Core functions for fetching high-resolution data from JHTDB in a single query, displaying progress, and saving snapshots.

In [ ]:
def fetch_full_plane(time: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fetch the full plane in a SINGLE query (orientation-aware).
    
    Args:
        time: Query time
    
    Returns:
        Tuple of (ux, uy, uz) arrays, each of shape (GRID_SIZE, GRID_SIZE)
    """
    # Create meshgrid for the entire domain
    cc1, cc2 = np.meshgrid(coord1_coords, coord2_coords, indexing='ij')
    
    # Create 3D point array based on plane orientation
    if plane_orientation == 'xz':
        # Wall-parallel (x-z plane, fixed y)
        points = np.column_stack([
            cc1.ravel(),                      # x coordinates
            np.full(cc1.size, fixed_coord),   # y = constant
            cc2.ravel()                        # z coordinates
        ])
    elif plane_orientation == 'xy':
        # Wall-normal (x-y plane, fixed z)
        points = np.column_stack([
            cc1.ravel(),                      # x coordinates
            cc2.ravel(),                      # y coordinates
            np.full(cc1.size, fixed_coord)    # z = constant
        ])
    elif plane_orientation == 'yz':
        # Wall-normal (y-z plane, fixed x)
        points = np.column_stack([
            np.full(cc1.size, fixed_coord),   # x = constant
            cc1.ravel(),                      # y coordinates
            cc2.ravel()                        # z coordinates
        ])
    
    # Query JHTDB (single query for all points!)
    result = getData(
        cube=dataset,
        var=VARIABLE,
        timepoint_original=time,
        temporal_method=TEMPORAL_METHOD,
        spatial_method_original=SPATIAL_METHOD,
        spatial_operator=SPATIAL_OPERATOR,
        points=points,
        verbose=False
    )
    
    # Extract velocity data
    velocity_df = result[0]
    velocity_array = velocity_df.to_numpy()  # Shape: (GRID_SIZE^2, 3)
    
    # Reshape to 2D grids
    velocity_reshaped = velocity_array.reshape(GRID_SIZE, GRID_SIZE, 3)
    
    ux_plane = velocity_reshaped[:, :, 0]
    uy_plane = velocity_reshaped[:, :, 1]
    uz_plane = velocity_reshaped[:, :, 2]
    
    return ux_plane, uy_plane, uz_plane


def format_time_for_filename(time: float) -> str:
    """
    Format time for filename, removing unnecessary trailing zeros.
    Examples: 0.1 -> '0.1', 0.12 -> '0.12', 0.125 -> '0.125', 1.0 -> '1.0'
    """
    # Round to 3 decimal places to avoid floating point issues
    rounded = round(time, 3)
    # Format with up to 3 decimal places, removing trailing zeros
    formatted = f"{rounded:.3f}".rstrip('0')
    # Ensure at least one decimal place remains
    if '.' in formatted and formatted.endswith('.'):
        formatted += '0'
    elif '.' not in formatted:
        formatted += '.0'
    return formatted


def save_snapshot_npz(time: float, ux: np.ndarray, uy: np.ndarray, uz: np.ndarray, omega_z: np.ndarray) -> Path:
    """
    Save snapshot as .npz file (orientation-aware).
    
    Args:
        time: Snapshot time
        ux, uy, uz: Velocity components
        omega_z: Vorticity
    
    Returns:
        Path to saved file
    """
    time_str = format_time_for_filename(time)
    
    # Generate filename based on plane orientation
    if plane_orientation == 'xz':
        filename = f"{DATASET}_{plane_orientation}_y{fixed_coord:.2f}_t{time_str}.npz"
        coord1_range = np.array([X_MIN, X_MAX])
        coord2_range = np.array([Z_MIN, Z_MAX])
    elif plane_orientation == 'xy':
        filename = f"{DATASET}_{plane_orientation}_z{fixed_coord:.2f}_t{time_str}.npz"
        coord1_range = np.array([X_MIN, X_MAX])
        coord2_range = np.array([Y_MIN, Y_MAX])
    elif plane_orientation == 'yz':
        filename = f"{DATASET}_{plane_orientation}_x{fixed_coord:.2f}_t{time_str}.npz"
        coord1_range = np.array([Y_MIN, Y_MAX])
        coord2_range = np.array([Z_MIN, Z_MAX])
    
    filepath = OUTPUT_DIR / filename
    
    np.savez(
        filepath,
        ux=ux,
        uy=uy,
        uz=uz,
        omega_z=omega_z,
        time=time,
        coord1_range=coord1_range,
        coord2_range=coord2_range,
        fixed_coord_name=fixed_coord_name,
        fixed_coord_value=fixed_coord,
        plane_orientation=plane_orientation,
        grid_size=GRID_SIZE
    )
    
    return filepath


def visualize_and_save_snapshot(time: float, ux: np.ndarray, uy: np.ndarray, 
                                uz: np.ndarray, omega_z: np.ndarray) -> Path:
    """
    Create and save visualization for a snapshot (orientation-aware).
    
    Args:
        time: Snapshot time
        ux, uy, uz: Velocity components
        omega_z: Vorticity
    
    Returns:
        Path to saved figure
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # Get colormap from config
    vel_cmap = VIZ_COLORMAPS.get('velocity', 'magma')
    vort_cmap = VIZ_COLORMAPS.get('vorticity', 'seismic')
    
    fields = [
        (ux, 'ux (streamwise)', vel_cmap),
        (uy, 'uy (wall-normal)', vel_cmap),
        (uz, 'uz (spanwise)', vel_cmap),
        (omega_z, f'ω_{fixed_coord_name} (vorticity)', vort_cmap)
    ]
    
    # Get extent based on plane orientation
    if plane_orientation == 'xz':
        extent = [X_MIN, X_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = 'x', 'z'
    elif plane_orientation == 'xy':
        extent = [X_MIN, X_MAX, Y_MIN, Y_MAX]
        xlabel, ylabel = 'x', 'y'
    elif plane_orientation == 'yz':
        extent = [Y_MIN, Y_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = 'y', 'z'
    
    for ax, (field, title, cmap) in zip(axes.flat, fields):
        im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                      extent=extent)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Add statistics
        stats = f'min={field.min():.3f}\nmax={field.max():.3f}\nmean={field.mean():.3f}'
        ax.text(0.02, 0.98, stats, transform=ax.transAxes, 
               verticalalignment='top', fontsize=8,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    time_str = format_time_for_filename(time)
    plt.suptitle(f'{DATASET} ({plane_orientation} plane): {GRID_SIZE}×{GRID_SIZE} | t={time_str}s | {fixed_coord_name}={fixed_coord:.2f}', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    figpath = VIZ_DIR / f"snapshot_t{time_str}.png"
    plt.savefig(figpath, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return figpath


print("✓ Fetching functions defined")

## Main Data Fetching Loop

Fetch all 1000 snapshots with progress tracking. Each snapshot will:
1. Fetch entire 512×512 field in **one query** (no tiling!)
2. Display the snapshot time
3. Save as `.npz` file in `data/raw/channel_sr_512/`
4. Save visualization in `data/raw/channel_sr_512/visualizations/`
5. Display visualization every 10 snapshots

In [ ]:
print(f"\n{'='*70}")
print(f"STARTING DATA FETCHING")
print(f"{'='*70}")
print(f"Dataset:           {DATASET}")
print(f"Plane:             {plane_orientation} ({coord1_label}-{coord2_label})")
print(f"Fixed {fixed_coord_name}: {fixed_coord:.4f}")
print(f"Grid size:         {GRID_SIZE}×{GRID_SIZE} = {GRID_SIZE*GRID_SIZE:,} points")
print(f"Snapshots:         {NUM_SNAPSHOTS}")
print(f"Time range:        {TIME_START:.2f} - {TIME_END:.2f} s")
print(f"Queries per snap:  1 (no tiling needed!)")
print(f"Total queries:     {NUM_SNAPSHOTS:,}")
print(f"Output:            {OUTPUT_DIR}")
print(f"{'='*70}\n")

failed_snapshots = []
saved_files = []

# Main loop
for idx, time in enumerate(tqdm(TIMES, desc="Fetching Snapshots", unit="snap")):
    try:
        # Fetch velocity field (single query!)
        ux, uy, uz = fetch_full_plane(float(time))
        
        # Compute vorticity using correct grid spacing
        omega_z = compute_vorticity_2d(ux, uy, dx=d_coord1, dy=d_coord2)
        
        # Save as .npz
        npz_path = save_snapshot_npz(float(time), ux, uy, uz, omega_z)
        saved_files.append(npz_path)
        
        # Create and save visualization
        viz_path = visualize_and_save_snapshot(float(time), ux, uy, uz, omega_z)
        
        # Display the visualization every N snapshots
        if (idx + 1) % SNAPSHOT_FREQUENCY == 0 or idx == 0:
            time_str = format_time_for_filename(float(time))
            print(f"\n✓ Snapshot {idx+1}/{NUM_SNAPSHOTS} (t={time_str}s) saved")
            print(f"  Data: {npz_path.name}")
            print(f"  Viz:  {viz_path.name}")
            
            # Display the figure inline
            display(HTML(f'<img src="{viz_path}" width="900">'))
        
    except Exception as e:
        print(f"\n✗ Error at snapshot {idx} (t={time:.3f}): {e}")
        failed_snapshots.append((idx, time, str(e)))
        continue

print(f"\n{'='*70}")
print(f"DATA FETCHING COMPLETE")
print(f"{'='*70}")
print(f"Successfully saved: {len(saved_files)}/{NUM_SNAPSHOTS} snapshots")
if failed_snapshots:
    print(f"Failed: {len(failed_snapshots)} snapshots")
    for idx, time, error in failed_snapshots[:5]:
        print(f"  - Snapshot {idx} (t={time:.3f}): {error}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Visualization directory: {VIZ_DIR}")
print(f"{'='*70}")

## Convert NPZ Files to HDF5

Consolidate all individual `.npz` files into a single HDF5 file for efficient loading in the SR pipeline.

In [ ]:
print("Converting .npz files to HDF5...\n")

# Find all .npz files for this configuration
npz_pattern = f"{DATASET}_{plane_orientation}_*.npz"
npz_files = sorted(OUTPUT_DIR.glob(npz_pattern))
print(f"Found {len(npz_files)} .npz files matching pattern: {npz_pattern}")

if len(npz_files) == 0:
    print("ERROR: No .npz files found! Please run the data fetching cell first.")
else:
    # Get HDF5 filename from config
    h5_filename = config['output'].get('h5_filename', f"{DATASET}_{plane_orientation}_{GRID_SIZE}.h5")
    h5_path = OUTPUT_DIR / h5_filename
    
    with h5py.File(h5_path, 'w') as h5f:
        # Create snapshots group
        h5f.create_group('snapshots')
        
        # Add global metadata
        h5f.attrs['dataset'] = DATASET
        h5f.attrs['variable'] = VARIABLE
        h5f.attrs['plane_orientation'] = plane_orientation
        h5f.attrs['plane_type'] = plane_type
        h5f.attrs['fixed_coord_name'] = fixed_coord_name
        h5f.attrs['fixed_coord_value'] = float(fixed_coord)
        h5f.attrs['grid_size'] = GRID_SIZE
        h5f.attrs['num_snapshots'] = len(npz_files)
        h5f.attrs['spatial_method'] = SPATIAL_METHOD
        h5f.attrs['created'] = datetime.now().isoformat()
        h5f.attrs['experiment_name'] = config['experiment']['name']
        h5f.attrs['description'] = config['experiment']['description']
        
        # Add domain info based on orientation
        if plane_orientation == 'xz':
            h5f.attrs['x_min'] = X_MIN
            h5f.attrs['x_max'] = X_MAX
            h5f.attrs['z_min'] = Z_MIN
            h5f.attrs['z_max'] = Z_MAX
            h5f.attrs['coord1_name'] = 'x'
            h5f.attrs['coord2_name'] = 'z'
        elif plane_orientation == 'xy':
            h5f.attrs['x_min'] = X_MIN
            h5f.attrs['x_max'] = X_MAX
            h5f.attrs['y_min'] = Y_MIN
            h5f.attrs['y_max'] = Y_MAX
            h5f.attrs['coord1_name'] = 'x'
            h5f.attrs['coord2_name'] = 'y'
        elif plane_orientation == 'yz':
            h5f.attrs['y_min'] = Y_MIN
            h5f.attrs['y_max'] = Y_MAX
            h5f.attrs['z_min'] = Z_MIN
            h5f.attrs['z_max'] = Z_MAX
            h5f.attrs['coord1_name'] = 'y'
            h5f.attrs['coord2_name'] = 'z'
        
        # Convert each .npz file
        for idx, npz_file in enumerate(tqdm(npz_files, desc="Converting")):
            # Load .npz
            data = np.load(npz_file)
            
            # Create snapshot group
            snap_name = f"{idx:04d}"
            grp = h5f['snapshots'].create_group(snap_name)
            
            # Save fields
            grp.create_dataset('ux', data=data['ux'].astype(np.float32), compression='gzip')
            grp.create_dataset('uy', data=data['uy'].astype(np.float32), compression='gzip')
            grp.create_dataset('uz', data=data['uz'].astype(np.float32), compression='gzip')
            grp.create_dataset('omega_z', data=data['omega_z'].astype(np.float32), compression='gzip')
            grp.create_dataset('time', data=np.float32(data['time']))
            
            # Add metadata
            grp.attrs['timestamp_index'] = idx
            grp.attrs['physical_time'] = float(data['time'])
            grp.attrs['fixed_coord_value'] = float(data['fixed_coord_value'])
            grp.attrs['grid_size'] = GRID_SIZE
            grp.attrs['plane_orientation'] = str(data['plane_orientation'])
    
    print(f"\n✓ HDF5 file created: {h5_path}")
    print(f"  File size: {h5_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Snapshots: {len(npz_files)}")
    print(f"  Grid size: {GRID_SIZE}×{GRID_SIZE}")
    print(f"  Plane: {plane_orientation} ({coord1_label}-{coord2_label})")
    print(f"  Fixed {fixed_coord_name}: {fixed_coord:.4f}")

## Batch Visualization: Load and Plot Multiple Snapshots

Load multiple snapshots specified by a time range and create **separate figures** for each velocity component and vorticity. Figures are saved to component-specific subdirectories.

In [ ]:
# Configuration for batch visualization
BATCH_T_START = 0.0      # Start time
BATCH_T_END = 5.0       # End time  
BATCH_NUM_SNAPS = 500    # Number of snapshots to visualize

# Vorticity colorbar settings
BATCH_VORTICITY_PERCENTILE = 99  # Percentile for vorticity (or None for full range)

# Generate time array
batch_times = np.linspace(BATCH_T_START, BATCH_T_END, BATCH_NUM_SNAPS)

# Create subdirectories for each component
component_dirs = {
    'velocity_x': VIZ_DIR / 'velocity_x',
    'velocity_y': VIZ_DIR / 'velocity_y',
    'velocity_z': VIZ_DIR / 'velocity_z',
    'vorticity_z': VIZ_DIR / 'vorticity_z'
}

for comp_dir in component_dirs.values():
    comp_dir.mkdir(parents=True, exist_ok=True)

print(f"Batch Visualization Configuration:")
print(f"  Time range: {BATCH_T_START} to {BATCH_T_END}")
print(f"  Number of snapshots: {BATCH_NUM_SNAPS}")
print(f"  Plane: {plane_orientation}, {fixed_coord_name}={fixed_coord:.2f}")
if BATCH_VORTICITY_PERCENTILE is not None:
    print(f"  Vorticity colorbar: {BATCH_VORTICITY_PERCENTILE}th percentile")
else:
    print(f"  Vorticity colorbar: Full range")
print(f"  Output directories:")
for name, path in component_dirs.items():
    print(f"    {name}: {path}")

# Get extent and labels based on plane orientation
if plane_orientation == 'xz':
    extent = [X_MIN, X_MAX, Z_MIN, Z_MAX]
    xlabel, ylabel = '$x$', '$z$'
    fixed_label = f'$y={fixed_coord:.2f}$'
elif plane_orientation == 'xy':
    extent = [X_MIN, X_MAX, Y_MIN, Y_MAX]
    xlabel, ylabel = '$x$ ', '$y$'
    fixed_label = f'$z={fixed_coord:.2f}$'
elif plane_orientation == 'yz':
    extent = [Y_MIN, Y_MAX, Z_MIN, Z_MAX]
    xlabel, ylabel = '$y$', '$z$'
    fixed_label = f'$x={fixed_coord:.2f}$'

# First pass: Calculate vorticity colorbar limits across all snapshots
if BATCH_VORTICITY_PERCENTILE is not None:
    print(f"\nCalculating vorticity colorbar limits...")
    all_vorticity = []
    for time in tqdm(batch_times[:10], desc="Sampling vorticity"):  # Sample first 10 for speed
        time_str = format_time_for_filename(float(time))
        
        # Construct filename
        if plane_orientation == 'xz':
            npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_y{fixed_coord:.2f}_t{time_str}.npz"
        elif plane_orientation == 'xy':
            npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_z{fixed_coord:.2f}_t{time_str}.npz"
        elif plane_orientation == 'yz':
            npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_x{fixed_coord:.2f}_t{time_str}.npz"
        
        if npz_filename.exists():
            data = np.load(npz_filename)
            all_vorticity.append(data['omega_z'].ravel())
    
    if all_vorticity:
        all_vorticity = np.concatenate(all_vorticity)
        vort_abs_max = np.percentile(np.abs(all_vorticity), BATCH_VORTICITY_PERCENTILE)
        vort_vmin, vort_vmax = -vort_abs_max, vort_abs_max
        print(f"  Vorticity limits: [±{vort_abs_max:.2f}] (Full range: [{all_vorticity.min():.2f}, {all_vorticity.max():.2f}])")
    else:
        vort_vmin, vort_vmax = None, None
else:
    vort_vmin, vort_vmax = None, None

# Load and plot each snapshot
print(f"\nProcessing {BATCH_NUM_SNAPS} snapshots...")

for time in tqdm(batch_times, desc="Creating component plots"):
    time_str = format_time_for_filename(float(time))
    
    # Construct filename based on plane orientation
    if plane_orientation == 'xz':
        npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_y{fixed_coord:.2f}_t{time_str}.npz"
    elif plane_orientation == 'xy':
        npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_z{fixed_coord:.2f}_t{time_str}.npz"
    elif plane_orientation == 'yz':
        npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_x{fixed_coord:.2f}_t{time_str}.npz"
    
    # Check if file exists
    if not npz_filename.exists():
        print(f"⚠ Warning: File not found: {npz_filename.name}")
        continue
    
    # Load data
    data = np.load(npz_filename)
    ux = data['ux']
    uy = data['uy']
    uz = data['uz']
    omega_z = data['omega_z']
    
    # Component specifications: (data, title, colormap, directory_key, vmin, vmax)
    components_to_plot = [
        (ux, 'Streamwise Velocity $u_x$', 'magma', 'velocity_x', None, None),
        (uy, 'Wall-Normal Velocity $u_y$', 'magma', 'velocity_y', None, None),
        (uz, 'Spanwise Velocity $u_z$', 'magma', 'velocity_z', None, None),
        (omega_z, f'Vorticity $\\omega_{{{fixed_coord_name}}}$', 'seismic', 'vorticity_z', vort_vmin, vort_vmax)
    ]
    
    # Create separate figure for each component
    for field, title, cmap, dir_key, vmin, vmax in components_to_plot:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot field with optional vmin/vmax
        if vmin is not None and vmax is not None:
            im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                          extent=extent, vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                          extent=extent)
        
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{title} at $t={time:.2f}$ s, {fixed_label}', 
                    fontsize=13, fontweight='bold')
        
        # Colorbar
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Add statistics text box
        stats = f'min: {field.min():.4f}\nmax: {field.max():.4f}\nmean: {field.mean():.4f}\nstd: {field.std():.4f}'
        if vmin is not None and vmax is not None:
            stats += f'\nCbar: [{vmin:.2f}, {vmax:.2f}]'
        ax.text(0.02, 0.98, stats, transform=ax.transAxes, 
               verticalalignment='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        
        plt.tight_layout()
        
        # Save figure
        save_path = component_dirs[dir_key] / f"{dir_key}_t{time_str}.png"
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close(fig)

print(f"\n✓ Batch visualization complete!")
print(f"  Created {BATCH_NUM_SNAPS * 4} individual component plots")
print(f"  Saved to: {VIZ_DIR}")
for name, path in component_dirs.items():
    num_files = len(list(path.glob('*.png')))
    print(f"    {name}: {num_files} files")

## Single Snapshot Visualization

Load and visualize a **single snapshot** with separate figures for each component (no saving, just display).

In [ ]:
# Specify the desired snapshot time
SINGLE_T = 10.0  # Time of the snapshot you want to visualize

# Colorbar limit settings for vorticity (to emphasize small-scale structures)
VORTICITY_PERCENTILE_CLIP = 99  # Use 98th percentile instead of max (1-99, or None for no clipping)
# Alternative: Set explicit limits (uncomment and set values, or set to None for auto)
# VORTICITY_VMIN = -50.0  # Set explicit minimum
# VORTICITY_VMAX = 50.0   # Set explicit maximum
VORTICITY_VMIN = None
VORTICITY_VMAX = None

# Generate filename based on current configuration
time_str = format_time_for_filename(SINGLE_T)

# Construct filename based on plane orientation
if plane_orientation == 'xz':
    npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_y{fixed_coord:.2f}_t{time_str}.npz"
elif plane_orientation == 'xy':
    npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_z{fixed_coord:.2f}_t{time_str}.npz"
elif plane_orientation == 'yz':
    npz_filename = OUTPUT_DIR / f"{DATASET}_{plane_orientation}_x{fixed_coord:.2f}_t{time_str}.npz"

print(f"Loading snapshot at t = {SINGLE_T} s")
print(f"  Configuration: {plane_orientation} plane, {fixed_coord_name}={fixed_coord:.2f}")
print(f"  File: {npz_filename.name}")

if not npz_filename.exists():
    print(f"\n✗ Error: File not found!")
    print(f"  Available time range: {TIME_START} to {TIME_END}")
    print(f"  Make sure the snapshot has been fetched.")
    print(f"  Expected pattern: {DATASET}_{plane_orientation}_*.npz")
else:
    # Load data
    data = np.load(npz_filename)
    ux = data['ux']
    uy = data['uy']
    uz = data['uz']
    omega_z = data['omega_z']
    actual_time = float(data['time'])
    
    print(f"✓ Loaded successfully!")
    print(f"  Actual time: {actual_time:.4f} s")
    print(f"  Grid size: {ux.shape}")
    
    # Get extent and labels based on plane orientation
    if plane_orientation == 'xz':
        extent = [X_MIN, X_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = '$x$', '$z$'
        domain_str = f"x ∈ [{X_MIN:.2f}, {X_MAX:.2f}], z ∈ [{Z_MIN:.2f}, {Z_MAX:.2f}]"
        fixed_str = f'$y={fixed_coord:.2f}$'
    elif plane_orientation == 'xy':
        extent = [X_MIN, X_MAX, Y_MIN, Y_MAX]
        xlabel, ylabel = '$x$', '$y$'
        domain_str = f"x ∈ [{X_MIN:.2f}, {X_MAX:.2f}], y ∈ [{Y_MIN:.2f}, {Y_MAX:.2f}]"
        fixed_str = f'$z={fixed_coord:.2f}$'
    elif plane_orientation == 'yz':
        extent = [Y_MIN, Y_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = '$y$', '$z$'
        domain_str = f"y ∈ [{Y_MIN:.2f}, {Y_MAX:.2f}], z ∈ [{Z_MIN:.2f}, {Z_MAX:.2f}]"
        fixed_str = f'$x={fixed_coord:.2f}$'
    
    print(f"  Domain: {domain_str}")
    
    # Calculate vorticity colorbar limits
    if VORTICITY_VMIN is not None and VORTICITY_VMAX is not None:
        # Use explicit user-defined limits
        vort_vmin, vort_vmax = VORTICITY_VMIN, VORTICITY_VMAX
        print(f"  Vorticity limits: Custom [{vort_vmin:.2f}, {vort_vmax:.2f}]")
    elif VORTICITY_PERCENTILE_CLIP is not None:
        # Use percentile-based clipping (symmetric around 0)
        vort_abs_max = np.percentile(np.abs(omega_z), VORTICITY_PERCENTILE_CLIP)
        vort_vmin, vort_vmax = -vort_abs_max, vort_abs_max
        print(f"  Vorticity limits: {VORTICITY_PERCENTILE_CLIP}th percentile [±{vort_abs_max:.2f}]")
        print(f"    (Full range: [{omega_z.min():.2f}, {omega_z.max():.2f}])")
    else:
        # Use full range (default)
        vort_vmin, vort_vmax = omega_z.min(), omega_z.max()
        print(f"  Vorticity limits: Full range [{vort_vmin:.2f}, {vort_vmax:.2f}]")
    
    # Component specifications: (field, title, cmap, vmin, vmax)
    components_to_plot = [
        (ux, 'Streamwise Velocity $u_x$', 'magma', None, None),
        (uy, 'Wall-Normal Velocity $u_y$', 'magma', None, None),
        (uz, 'Spanwise Velocity $u_z$', 'magma', None, None),
        (omega_z, f'Vorticity $\\omega_{{{fixed_coord_name}}}$', 'seismic', vort_vmin, vort_vmax)
    ]
    
    # Create and display separate figures for each component
    for field, title, cmap, vmin, vmax in components_to_plot:
        fig, ax = plt.subplots(figsize=(12, 7), dpi=180)
        
        # Plot field with optional vmin/vmax
        if vmin is not None and vmax is not None:
            im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                          extent=extent, vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                          extent=extent)
        
        ax.set_xlabel(xlabel, fontsize=13)
        ax.set_ylabel(ylabel, fontsize=13)
        ax.set_title(f'{title} at $t={actual_time:.3f}$ s, {fixed_str}', 
                    fontsize=14, fontweight='bold')
        
        # Colorbar
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=11)
        
        # Add detailed statistics
        stats_text = (
            f'Statistics:\n'
            f'  min:  {field.min():.5f}\n'
            f'  max:  {field.max():.5f}\n'
            f'  mean: {field.mean():.5f}\n'
            f'  std:  {field.std():.5f}\n'
            f'  median: {np.median(field):.5f}'
        )
        
        # Add colorbar limit info for vorticity
        if vmin is not None and vmax is not None:
            stats_text += f'\nColorbar: [{vmin:.3f}, {vmax:.3f}]'
        
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
               verticalalignment='top', fontsize=10,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.95, edgecolor='black'),
               family='monospace')
        
        plt.tight_layout()
        plt.show()
    
    print(f"\n✓ Displayed all 4 component figures")

## Create Animations

Generate animated GIFs for each velocity component and vorticity to visualize temporal evolution.

In [ ]:
print("Creating animations...\n")

# Vorticity colorbar settings for animations
ANIM_VORTICITY_PERCENTILE = 99  # Percentile for vorticity colorbar (or None for full range)

# Find all .npz files for this configuration and sort by time
npz_pattern = f"{DATASET}_{plane_orientation}_*.npz"
npz_files = sorted(OUTPUT_DIR.glob(npz_pattern), 
                   key=lambda x: float(x.stem.split('_t')[-1]))

if len(npz_files) == 0:
    print(f"ERROR: No .npz files found matching pattern: {npz_pattern}")
    print("Please run the data fetching cell first.")
else:
    # Load all snapshots
    print(f"Loading {len(npz_files)} snapshots...")
    snapshots = []
    times = []
    
    for npz_file in tqdm(npz_files, desc="Loading"):
        data = np.load(npz_file)
        snapshots.append({
            'ux': data['ux'],
            'uy': data['uy'],
            'uz': data['uz'],
            'omega_z': data['omega_z'],
        })
        times.append(float(data['time']))
    
    # Get extent and labels based on plane orientation
    if plane_orientation == 'xz':
        extent = [X_MIN, X_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = 'x', 'z'
        fixed_label = f'y={fixed_coord:.2f}'
    elif plane_orientation == 'xy':
        extent = [X_MIN, X_MAX, Y_MIN, Y_MAX]
        xlabel, ylabel = 'x', 'y'
        fixed_label = f'z={fixed_coord:.2f}'
    elif plane_orientation == 'yz':
        extent = [Y_MIN, Y_MAX, Z_MIN, Z_MAX]
        xlabel, ylabel = 'y', 'z'
        fixed_label = f'x={fixed_coord:.2f}'
    
    # Create animations for each component
    components = [
        ('ux', 'Streamwise Velocity (ux)', 'magma', False),
        ('uy', 'Wall-normal Velocity (uy)', 'magma', False),
        ('uz', 'Spanwise Velocity (uz)', 'magma', False),
        ('omega_z', f'Vorticity (ω_{fixed_coord_name})', 'seismic', True),
    ]
    
    for comp_name, comp_title, cmap, is_vorticity in components:
        print(f"\nCreating animation for {comp_name}...")
        
        # Collect all data for this component
        data_list = [snap[comp_name] for snap in snapshots]
        
        # Determine consistent colorbar limits
        if is_vorticity and ANIM_VORTICITY_PERCENTILE is not None:
            # Use percentile clipping for vorticity
            all_data = np.concatenate([d.ravel() for d in data_list])
            vort_abs_max = np.percentile(np.abs(all_data), ANIM_VORTICITY_PERCENTILE)
            vmin, vmax = -vort_abs_max, vort_abs_max
            print(f"  Using {ANIM_VORTICITY_PERCENTILE}th percentile: [±{vort_abs_max:.2f}]")
            print(f"  (Full range: [{all_data.min():.2f}, {all_data.max():.2f}])")
        else:
            # Use full range for velocity or if percentile is None
            vmin = min(d.min() for d in data_list)
            vmax = max(d.max() for d in data_list)
            if is_vorticity:
                print(f"  Using full range: [{vmin:.2f}, {vmax:.2f}]")
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Initial plot
        im = ax.imshow(data_list[0].T, origin='lower', cmap=cmap, 
                      aspect='auto', vmin=vmin, vmax=vmax,
                      extent=extent)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        title = ax.set_title(f'{comp_title}\nt={times[0]:.2f}s, {fixed_label}', 
                            fontweight='bold')
        
        def update(frame):
            im.set_array(data_list[frame].T)
            title.set_text(f'{comp_title}\nt={times[frame]:.2f}s, {fixed_label}')
            return [im, title]
        
        # Create animation
        anim = FuncAnimation(fig, update, frames=len(data_list), 
                           interval=100, blit=True)
        
        # Save as GIF
        gif_path = VIZ_DIR / f"animation_{comp_name}.gif"
        writer = PillowWriter(fps=10)
        anim.save(gif_path, writer=writer)
        plt.close(fig)
        
        print(f"  ✓ Saved: {gif_path.name}")
        print(f"    Frames: {len(data_list)}, Size: {gif_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    print(f"\n{'='*70}")
    print(f"ANIMATIONS COMPLETE")
    print(f"{'='*70}")
    print(f"Location: {VIZ_DIR}")
    print(f"Files:")
    for gif_file in sorted(VIZ_DIR.glob("animation_*.gif")):
        print(f"  - {gif_file.name}")
    print(f"{'='*70}")

## Summary

Data fetching pipeline completed successfully!

In [ ]:
print(f"\n{'='*70}")
print(f"JHTDB HIGH-RESOLUTION DATA FETCHING SUMMARY")
print(f"{'='*70}")
print(f"Dataset:           {DATASET}")
print(f"Grid resolution:   {GRID_SIZE}×{GRID_SIZE} ({GRID_SIZE*GRID_SIZE:,} points)")
print(f"Fixed coordinate:   {fixed_coord_name} = {fixed_coord_value:.4f}")
print(f"Snapshots:         {len(npz_files)}")
print(f"Time range:        {TIME_START:.2f} - {TIME_END:.2f} s")
print(f"Time spacing:      {(TIME_END - TIME_START) / (NUM_SNAPSHOTS - 1):.4f} s")
print(f"\nOutput files:")
print(f"  NPZ files:       {OUTPUT_DIR}")
print(f"  HDF5 file:       {OUTPUT_DIR / config['output']['h5_filename']}")
print(f"  Visualizations:  {VIZ_DIR}")
print(f"  Animations:      {len(list(VIZ_DIR.glob('animation_*.gif')))} GIF files")
print(f"\nKey features:")
print(f"  ✓ Single-query fetching (no tiling)")
print(f"  ✓ {len(npz_files)} high-resolution snapshots")
print(f"  ✓ Ready for super-resolution training")
print(f"\nNext step: Use this data in ml_pipeline_sr.ipynb for training!")
print(f"{'='*70}")

## Alternative: Tiling-Based Approach for Old Token (4,096 Point Limit)

**For users with the standard JHTDB token (4,096 point limit)**, this section provides an alternative tiling-based fetching approach. This is useful for:
- Open-source users without access to high-capacity tokens
- Fetching 256×256 grids (requires 16 tiles)
- Understanding how to work within the 4,096 point constraint

**How tiling works:**
1. Split the domain into smaller tiles (e.g., 64×64 = 4,096 points)
2. Fetch each tile separately
3. Stitch tiles together to form the full field

**Example use case:** Fetch 100 snapshots of 256×256 channel flow data using 16 tiles per snapshot.

### Alternative Configuration (256×256 with Tiling)

In [ ]:
# Alternative configuration for 256×256 with standard token
ALT_GRID_SIZE = 256                     # Grid resolution
ALT_TILE_SIZE = 64                      # Tile size (64×64 = 4,096 points)
ALT_NUM_SNAPSHOTS = 100                 # Number of snapshots
ALT_TIME_START = 0.1
ALT_TIME_END = 25.0
ALT_TIMES = np.linspace(ALT_TIME_START, ALT_TIME_END, ALT_NUM_SNAPSHOTS)

# Output directory (different from high-res)
ALT_OUTPUT_DIR = ROOT_DIR / "data/raw/channel_sr_256"
ALT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALT_VIZ_DIR = ALT_OUTPUT_DIR / "visualizations"
ALT_VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Create an orientation-aware alternative grid. The legacy helper names
# alt_x_coords/alt_z_coords are retained for the optional cells below.
if plane_orientation == 'xz':
    alt_coord1_min, alt_coord1_max = X_MIN, X_MAX
    alt_coord2_min, alt_coord2_max = Z_MIN, Z_MAX
elif plane_orientation == 'xy':
    alt_coord1_min, alt_coord1_max = X_MIN, X_MAX
    alt_coord2_min, alt_coord2_max = Y_MIN, Y_MAX
else:
    alt_coord1_min, alt_coord1_max = Y_MIN, Y_MAX
    alt_coord2_min, alt_coord2_max = Z_MIN, Z_MAX
alt_x_coords = np.linspace(alt_coord1_min, alt_coord1_max, ALT_GRID_SIZE, dtype=np.float64)
alt_z_coords = np.linspace(alt_coord2_min, alt_coord2_max, ALT_GRID_SIZE, dtype=np.float64)
alt_dx = alt_x_coords[1] - alt_x_coords[0]
alt_dz = alt_z_coords[1] - alt_z_coords[0]

# Create tile boundaries
num_tiles_per_dim = ALT_GRID_SIZE // ALT_TILE_SIZE
tile_boundaries = []
for i in range(num_tiles_per_dim):
    for j in range(num_tiles_per_dim):
        x_start, x_end = i * ALT_TILE_SIZE, (i + 1) * ALT_TILE_SIZE
        z_start, z_end = j * ALT_TILE_SIZE, (j + 1) * ALT_TILE_SIZE
        tile_boundaries.append((x_start, x_end, z_start, z_end))

print(f"Alternative Configuration (Standard Token):")
print(f"  Grid: {ALT_GRID_SIZE}×{ALT_GRID_SIZE} = {ALT_GRID_SIZE*ALT_GRID_SIZE:,} points")
print(f"  Tile size: {ALT_TILE_SIZE}×{ALT_TILE_SIZE} = {ALT_TILE_SIZE*ALT_TILE_SIZE:,} points")
print(f"  Number of tiles: {len(tile_boundaries)} ({num_tiles_per_dim}×{num_tiles_per_dim})")
print(f"  Snapshots: {ALT_NUM_SNAPSHOTS}")
print(f"  Output: {ALT_OUTPUT_DIR}")

### Alternative Fetching Functions (Tiling-Based)

In [ ]:
def alt_fetch_tile(time: float, x_start: int, x_end: int, z_start: int, z_end: int) -> np.ndarray:
    """
    Fetch a single tile of velocity data from JHTDB.
    
    Args:
        time: Query time
        x_start, x_end: X-index range
        z_start, z_end: Z-index range
    
    Returns:
        Velocity array of shape (num_points, 3) where columns are [ux, uy, uz]
    """
    # Extract coordinates for this tile
    x_tile = alt_x_coords[x_start:x_end]
    z_tile = alt_z_coords[z_start:z_end]
    
    # Create meshgrid and flatten to point array
    xx, zz = np.meshgrid(x_tile, z_tile, indexing='ij')
    points = np.column_stack([
        xx.ravel(),
        np.full(xx.size, y_coord),
        zz.ravel()
    ])
    
    # Query JHTDB
    result = getData(
        cube=dataset,
        var=VARIABLE,
        timepoint_original=time,
        temporal_method=TEMPORAL_METHOD,
        spatial_method_original=SPATIAL_METHOD,
        spatial_operator=SPATIAL_OPERATOR,
        points=points,
        verbose=False
    )
    
    # Extract velocity data
    velocity_df = result[0]
    velocity_array = velocity_df.to_numpy()
    
    return velocity_array


def alt_fetch_full_plane_with_progress(time: float, idx: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fetch the full 256×256 plane with tile-by-tile progress display.
    
    Args:
        time: Query time
        idx: Snapshot index
    
    Returns:
        Tuple of (ux, uy, uz) arrays, each of shape (ALT_GRID_SIZE, ALT_GRID_SIZE)
    """
    # Initialize arrays
    ux_plane = np.zeros((ALT_GRID_SIZE, ALT_GRID_SIZE), dtype=np.float32)
    uy_plane = np.zeros((ALT_GRID_SIZE, ALT_GRID_SIZE), dtype=np.float32)
    uz_plane = np.zeros((ALT_GRID_SIZE, ALT_GRID_SIZE), dtype=np.float32)
    
    # Progress bar for tiles
    desc = f"Snapshot {idx+1}/{ALT_NUM_SNAPSHOTS} (t={time:.3f}s)"
    tile_pbar = tqdm(tile_boundaries, desc=desc, leave=False, unit='tile')
    
    for tile_idx, (x_start, x_end, z_start, z_end) in enumerate(tile_pbar):
        # Update progress with tile coordinates
        x_range = (alt_x_coords[x_start], alt_x_coords[x_end-1])
        z_range = (alt_z_coords[z_start], alt_z_coords[z_end-1])
        tile_pbar.set_postfix({
            'tile': f'{tile_idx+1}/{len(tile_boundaries)}',
            'x': f'[{x_range[0]:.2f},{x_range[1]:.2f}]',
            'z': f'[{z_range[0]:.2f},{z_range[1]:.2f}]'
        })
        
        # Fetch tile
        velocity = alt_fetch_tile(time, x_start, x_end, z_start, z_end)
        
        # Reshape and place in full arrays
        tile_shape = (x_end - x_start, z_end - z_start)
        velocity_reshaped = velocity.reshape(tile_shape[0], tile_shape[1], 3)
        
        ux_plane[x_start:x_end, z_start:z_end] = velocity_reshaped[:, :, 0]
        uy_plane[x_start:x_end, z_start:z_end] = velocity_reshaped[:, :, 1]
        uz_plane[x_start:x_end, z_start:z_end] = velocity_reshaped[:, :, 2]
    
    tile_pbar.close()
    return ux_plane, uy_plane, uz_plane


def alt_save_snapshot_npz(time: float, ux: np.ndarray, uy: np.ndarray, 
                          uz: np.ndarray, omega_z: np.ndarray) -> Path:
    """
    Save snapshot as .npz file in data/raw/channel_sr_256/
    
    Args:
        time: Snapshot time
        ux, uy, uz: Velocity components
        omega_z: Vorticity
    
    Returns:
        Path to saved file
    """
    time_str = format_time_for_filename(time)
    filename = f"channel_x-z_y{Y_LOCATION}_t{time_str}.npz"
    filepath = ALT_OUTPUT_DIR / filename
    
    np.savez(
        filepath,
        ux=ux,
        uy=uy,
        uz=uz,
        omega_z=omega_z,
        time=time,
        x_range=np.array([X_MIN, X_MAX]),
        z_range=np.array([Z_MIN, Z_MAX]),
        y_value=Y_LOCATION,
        grid_size=ALT_GRID_SIZE
    )
    
    return filepath


def alt_visualize_and_save_snapshot(time: float, ux: np.ndarray, uy: np.ndarray, 
                                    uz: np.ndarray, omega_z: np.ndarray) -> Path:
    """
    Create and save visualization for a snapshot.
    
    Args:
        time: Snapshot time
        ux, uy, uz: Velocity components
        omega_z: Vorticity
    
    Returns:
        Path to saved figure
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    fields = [
        (ux, 'ux (streamwise)', 'RdBu_r'),
        (uy, 'uy (wall-normal)', 'RdBu_r'),
        (uz, 'uz (spanwise)', 'RdBu_r'),
        (omega_z, 'ωz (vorticity)', 'seismic')
    ]
    
    for ax, (field, title, cmap) in zip(axes.flat, fields):
        im = ax.imshow(field.T, origin='lower', cmap=cmap, aspect='auto',
                      extent=[X_MIN, X_MAX, Z_MIN, Z_MAX])
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('x')
        ax.set_ylabel('z')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Add statistics
        stats = f'min={field.min():.3f}\nmax={field.max():.3f}\nmean={field.mean():.3f}'
        ax.text(0.02, 0.98, stats, transform=ax.transAxes, 
               verticalalignment='top', fontsize=7,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    time_str = format_time_for_filename(time)
    plt.suptitle(f'Snapshot (Tiled): {ALT_GRID_SIZE}×{ALT_GRID_SIZE} | t={time_str}s | y={Y_LOCATION}', 
                fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    figpath = ALT_VIZ_DIR / f"snapshot_t{time_str}.png"
    plt.savefig(figpath, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    return figpath


print("✓ Alternative tiling-based fetching functions defined")

### Alternative Main Loop (Tiling-Based)

**To use this alternative approach**, uncomment and run the cell below. This will fetch 100 snapshots of 256×256 data using the tiling approach (16 queries per snapshot).

In [ ]:
# Uncomment to run the alternative tiling-based fetching
"""
print(f"\n{'='*70}")
print(f"ALTERNATIVE: TILING-BASED DATA FETCHING (256×256)")
print(f"{'='*70}")
print(f"Grid size:         {ALT_GRID_SIZE}×{ALT_GRID_SIZE} = {ALT_GRID_SIZE*ALT_GRID_SIZE:,} points")
print(f"Tile size:         {ALT_TILE_SIZE}×{ALT_TILE_SIZE} = {ALT_TILE_SIZE*ALT_TILE_SIZE:,} points")
print(f"Tiles per snap:    {len(tile_boundaries)}")
print(f"Snapshots:         {ALT_NUM_SNAPSHOTS}")
print(f"Time range:        {ALT_TIME_START:.2f} - {ALT_TIME_END:.2f} s")
print(f"Total queries:     {ALT_NUM_SNAPSHOTS * len(tile_boundaries):,}")
print(f"Output:            {ALT_OUTPUT_DIR}")
print(f"{'='*70}\n")

alt_failed_snapshots = []
alt_saved_files = []

# Main loop
for idx, time in enumerate(tqdm(ALT_TIMES, desc="Fetching Snapshots (Tiled)", unit="snap")):
    try:
        # Fetch velocity field with tiling progress
        ux, uy, uz = alt_fetch_full_plane_with_progress(float(time), idx)
        
        # Compute vorticity
        omega_z = compute_vorticity_2d(ux, uy, dx=alt_dx, dy=alt_dz)
        
        # Save as .npz
        npz_path = alt_save_snapshot_npz(float(time), ux, uy, uz, omega_z)
        alt_saved_files.append(npz_path)
        
        # Create and save visualization
        viz_path = alt_visualize_and_save_snapshot(float(time), ux, uy, uz, omega_z)
        
        # Display the visualization every 10 snapshots
        if (idx + 1) % 10 == 0 or idx == 0:
            time_str = format_time_for_filename(float(time))
            print(f"\n✓ Snapshot {idx+1}/{ALT_NUM_SNAPSHOTS} (t={time_str}s) saved")
            print(f"  Data: {npz_path.name}")
            print(f"  Viz:  {viz_path.name}")
            
            # Display the figure inline
            display(HTML(f'<img src="{viz_path}" width="800">'))
        
    except Exception as e:
        print(f"\n✗ Error at snapshot {idx} (t={time:.3f}): {e}")
        alt_failed_snapshots.append((idx, time, str(e)))
        continue

print(f"\n{'='*70}")
print(f"TILING-BASED DATA FETCHING COMPLETE")
print(f"{'='*70}")
print(f"Successfully saved: {len(alt_saved_files)}/{ALT_NUM_SNAPSHOTS} snapshots")
if alt_failed_snapshots:
    print(f"Failed: {len(alt_failed_snapshots)} snapshots")
    for idx, time, error in alt_failed_snapshots[:5]:
        print(f"  - Snapshot {idx} (t={time:.3f}): {error}")
print(f"\nOutput directory: {ALT_OUTPUT_DIR}")
print(f"Visualization directory: {ALT_VIZ_DIR}")
print(f"{'='*70}")
"""

print("Alternative tiling-based loop ready (currently commented out)")
print("Uncomment the cell above to use 256×256 tiling-based fetching with standard token")